# Statistical Feature Research

This notebook develops an independent statistical research branch for GC/MGC. This first increment is deliberately limited to research-environment configuration and validation of the trusted active-contract table. It does not load raw DBN data or POI outputs, rebuild upstream pipelines, construct observations or labels, or engineer features.

# 1.0 Configuration and Research Environment

This section establishes a reproducible project environment and locks the market-time conventions that later research must obey. All paths are resolved relative to the project root, UTC timestamps remain reference keys, and New York time is authoritative for every trading-session rule.

## 1.1 Imports

Only standard numerical, tabular, timezone, and Parquet dependencies are imported at this stage. Package versions and the running executable are printed so that the executed notebook records its research environment.

In [1]:
from __future__ import annotations

import random
import re
import sys
from datetime import time
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

print(f"Python {sys.version.split()[0]}")
print(f"Executable: {sys.executable}")
print(f"pandas {pd.__version__} | numpy {np.__version__} | pyarrow {pa.__version__}")

Python 3.14.5
Executable: C:\Users\abond\Desktop\WORK FILES\Systemic\Project 1\.venv-1\Scripts\python.exe
pandas 3.0.3 | numpy 2.4.6 | pyarrow 24.0.0


## 1.2 Paths

The project root is discovered by walking upward from the current working directory until both trusted project markers are found. The notebook therefore works whether launched from the project root or a notebook subdirectory. The active interpreter is required to be the project's `.venv-1` environment.

In [2]:
TRUSTED_DATA_RELATIVE_PATH = Path("data/processed/research_bars_gc_mgc_1m.parquet")
CONTEXT_RELATIVE_PATH = Path("project_docs/statistical_feature_research_context_report.md")
NOTEBOOK_RELATIVE_PATH = Path("notebooks/exploration/statistical_feature_research.ipynb")


def find_project_root(start: Path | None = None) -> Path:
    """Locate the project without relying on the notebook's launch directory."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / TRUSTED_DATA_RELATIVE_PATH).is_file() and (candidate / CONTEXT_RELATIVE_PATH).is_file():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / TRUSTED_DATA_RELATIVE_PATH
CONTEXT_REPORT_PATH = PROJECT_ROOT / CONTEXT_RELATIVE_PATH
NOTEBOOK_PATH = PROJECT_ROOT / NOTEBOOK_RELATIVE_PATH
EXPECTED_PYTHON = PROJECT_ROOT / ".venv-1" / "Scripts" / "python.exe"

assert DATA_PATH.is_file(), f"Missing trusted dataset: {DATA_PATH}"
assert CONTEXT_REPORT_PATH.is_file(), f"Missing governing report: {CONTEXT_REPORT_PATH}"
assert DATA_PATH.name == "research_bars_gc_mgc_1m.parquet" and DATA_PATH.suffix == ".parquet"
assert not any(token in DATA_PATH.name.lower() for token in (".dbn", "section6_", "section7_", "poi"))
assert Path(sys.executable).resolve() == EXPECTED_PYTHON.resolve(), (
    f"Wrong interpreter. Expected {EXPECTED_PYTHON}, received {sys.executable}."
)

environment_check = pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "trusted_data": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "governing_context": str(CONTEXT_REPORT_PATH.relative_to(PROJECT_ROOT)),
        "active_interpreter": str(Path(sys.executable).resolve()),
        "expected_environment": ".venv-1",
        "environment_valid": True,
    },
    name="value",
)
display(environment_check.to_frame())

,value
project_root,C:\Users\abond\Desktop\WORK FILES\Systemic\Pro...
trusted_data,data\processed\research_bars_gc_mgc_1m.parquet
governing_context,project_docs\statistical_feature_research_cont...
active_interpreter,C:\Users\abond\Desktop\WORK FILES\Systemic\Pro...
expected_environment,.venv-1
environment_valid,True


## 1.3 Instrument and Time Settings

GC and MGC are the only expected products, with GC designated for the later Phase 1 discovery population. `America/New_York` is the authoritative timezone for session classification, trading dates, entry eligibility, the noon cutoff, and the 3:30pm mandatory exit. UTC is retained only as an immutable timestamp reference. The entry intervals are recorded now but are not applied until the eligible observation frame is constructed in a later section.

In [3]:
EXPECTED_PRODUCTS = ("GC", "MGC")
PRIMARY_RESEARCH_PRODUCT = "GC"
RESEARCH_TIMEZONE_NAME = "America/New_York"
REFERENCE_TIMEZONE_NAME = "UTC"
RESEARCH_TIMEZONE = ZoneInfo(RESEARCH_TIMEZONE_NAME)
REFERENCE_TIMEZONE = ZoneInfo(REFERENCE_TIMEZONE_NAME)
UTC_REFERENCE_KEY_COLUMNS = ("ts_event_utc", "product")

ENTRY_WINDOWS_NY = {
    "London": (time(3, 0), time(6, 0)),
    "New York": (time(7, 0), time(12, 0)),
}
NO_NEW_ENTRY_AT_OR_AFTER_NY = time(12, 0)
MANDATORY_EXIT_NY = time(15, 30)

assert PRIMARY_RESEARCH_PRODUCT in EXPECTED_PRODUCTS
assert RESEARCH_TIMEZONE.key == "America/New_York"
assert ENTRY_WINDOWS_NY["London"] == (time(3, 0), time(6, 0))
assert ENTRY_WINDOWS_NY["New York"] == (time(7, 0), time(12, 0))
assert NO_NEW_ENTRY_AT_OR_AFTER_NY == time(12, 0)
assert MANDATORY_EXIT_NY == time(15, 30)

time_settings = pd.Series(
    {
        "expected_products": EXPECTED_PRODUCTS,
        "phase_1_product": PRIMARY_RESEARCH_PRODUCT,
        "research_timezone": RESEARCH_TIMEZONE_NAME,
        "UTC_reference_keys": UTC_REFERENCE_KEY_COLUMNS,
        "London_entry_window_NY": "[03:00, 06:00)",
        "New_York_entry_window_NY": "[07:00, 12:00)",
        "mandatory_exit_NY": "15:30",
    },
    name="setting",
)
display(time_settings.to_frame())

,setting
expected_products,"(GC, MGC)"
phase_1_product,GC
research_timezone,America/New_York
UTC_reference_keys,"(ts_event_utc, product)"
London_entry_window_NY,"[03:00, 06:00)"
New_York_entry_window_NY,"[07:00, 12:00)"
mandatory_exit_NY,15:30


## 1.4 Random Seeds and Display Settings

A fixed seed is set for reproducible sampling and later statistical procedures. Display settings expose enough rows and columns to audit the saved schema and validation tables without changing analytical results.

In [4]:
RANDOM_SEED = 20260714
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

print(f"Random seed fixed at {RANDOM_SEED}.")

Random seed fixed at 20260714.


# 2.0 Load Trusted Research Data

The sole market-data input is the completed active-contract research table. The saved Arrow schema is inspected before the table is materialized; no raw DBN archive, upstream active-contract logic, or Section 6/7 POI output is read.

## 2.1 Load Final GC/MGC Research Table

Parquet metadata and the physical Arrow schema are read first. The complete table is then loaded once with Arrow-backed pandas dtypes to reduce memory overhead while preserving the saved timezone and integer types. Subsequent checks reuse this single dataframe.

In [5]:
parquet_file = pq.ParquetFile(DATA_PATH)
saved_arrow_schema = parquet_file.schema_arrow
saved_column_names = tuple(saved_arrow_schema.names)
parquet_metadata = pd.Series(
    {
        "relative_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "rows": parquet_file.metadata.num_rows,
        "columns": parquet_file.metadata.num_columns,
        "row_groups": parquet_file.metadata.num_row_groups,
        "file_size_MiB": DATA_PATH.stat().st_size / (1024**2),
    },
    name="saved_value",
)
display(parquet_metadata.to_frame())

research_bars = pd.read_parquet(DATA_PATH, engine="pyarrow", dtype_backend="pyarrow")

assert len(research_bars) == parquet_file.metadata.num_rows
assert tuple(research_bars.columns) == saved_column_names
loaded_summary = pd.Series(
    {
        "loaded_rows": len(research_bars),
        "loaded_columns": research_bars.shape[1],
        "dataframe_memory_MiB": research_bars.memory_usage(index=True, deep=True).sum() / (1024**2),
        "dtype_backend": "pyarrow",
    },
    name="loaded_value",
)
display(loaded_summary.to_frame())

,saved_value
relative_path,data\processed\research_bars_gc_mgc_1m.parquet
rows,3487656
columns,82
row_groups,4
file_size_MiB,603.389116


,loaded_value
loaded_rows,3487656
loaded_columns,82
dataframe_memory_MiB,"1,890.141650"
dtype_backend,pyarrow


## 2.2 Confirm Schema and Data Types

The saved Arrow type and loaded pandas type are shown for every column before any downstream assumptions are made. Required identifiers, market fields, session fields, and controls are asserted. Product counts, within-product timestamp order, UTC-key uniqueness, and core OHLCV completeness are validated.

Existing columns whose names indicate a future market outcome are placed in `EXISTING_FORWARD_COLUMNS`. This quarantine is a namespace safeguard: these columns remain available only for possible later validation and must never enter the independent feature frame. Operational schedule fields such as `days_to_next_roll` are rollover controls, not realized market outcomes, and are not classified as feature labels here. Timestamp sorting is required within each product; no assumption is made about the panel's physical interleaving across products.

In [6]:
schema_table = pd.DataFrame(
    {
        "column": saved_column_names,
        "saved_arrow_type": [str(field.type) for field in saved_arrow_schema],
        "loaded_pandas_type": [str(research_bars.dtypes[name]) for name in saved_column_names],
    }
)
display(schema_table)

REQUIRED_COLUMNS = {
    "ts_event_utc", "ts_event_ny", "trade_date_utc", "trade_date_ny",
    "product", "symbol", "active_symbol", "open", "high", "low", "close", "volume",
    "hour_ny", "minute_ny", "session_label", "continuous_segment_id",
    "is_roll_day", "is_pre_roll_day", "is_post_roll_day", "roll_window_flag",
    "roll_window_type", "days_since_roll", "days_to_next_roll", "tradable_research_flag",
}
CORE_OHLCV_COLUMNS = ["open", "high", "low", "close", "volume"]
EXPECTED_ROW_COUNTS = {"GC": 1_759_671, "MGC": 1_727_985}
OUTCOME_NAME_PATTERNS = (
    r"(^|_)forward(_|$)",
    r"(^|_)future(_|$)",
    r"(^|_)(mfe|mae)(_|$)",
    r"(^|_)(target|stop).*(hit|outcome)",
    r"(^|_)outcome(_|$)",
)
EXISTING_FORWARD_COLUMNS = [
    name
    for name in saved_column_names
    if any(re.search(pattern, name.lower()) for pattern in OUTCOME_NAME_PATTERNS)
]

missing_required_columns = sorted(REQUIRED_COLUMNS.difference(research_bars.columns))
product_counts = research_bars["product"].value_counts(sort=False).sort_index()
per_product_timestamp_sorted = {
    product: research_bars.loc[research_bars["product"].eq(product), "ts_event_utc"].is_monotonic_increasing
    for product in EXPECTED_PRODUCTS
}
global_timestamp_sorted = research_bars["ts_event_utc"].is_monotonic_increasing
duplicate_timestamp_product_rows = int(research_bars.duplicated(list(UTC_REFERENCE_KEY_COLUMNS)).sum())
core_ohlcv_missing = research_bars[CORE_OHLCV_COLUMNS].isna().sum()

schema_checks = pd.Series(
    {
        "all_required_columns_present": not missing_required_columns,
        "expected_products_only": set(product_counts.index) == set(EXPECTED_PRODUCTS),
        "expected_product_row_counts": product_counts.to_dict() == EXPECTED_ROW_COUNTS,
        "timestamps_sorted_within_each_product": all(per_product_timestamp_sorted.values()),
        "duplicate_timestamp_product_rows_zero": duplicate_timestamp_product_rows == 0,
        "core_OHLCV_missing_values_zero": int(core_ohlcv_missing.sum()) == 0,
        "existing_forward_columns_quarantined": len(EXISTING_FORWARD_COLUMNS) > 0,
    },
    name="passed",
)

display(product_counts.rename("rows").to_frame())
display(pd.DataFrame({"timestamp_sorted": per_product_timestamp_sorted}))
display(core_ohlcv_missing.rename("missing_values").to_frame())
print(f"Global physical timestamp order (informational): {global_timestamp_sorted}")
print(f"Duplicate (ts_event_utc, product) rows: {duplicate_timestamp_product_rows:,}")
print(f"EXISTING_FORWARD_COLUMNS ({len(EXISTING_FORWARD_COLUMNS)}): {EXISTING_FORWARD_COLUMNS}")
display(schema_checks.to_frame())

assert schema_checks.all(), schema_checks[~schema_checks].index.tolist()

,column,saved_arrow_type,loaded_pandas_type
0,ts_event_utc,"timestamp[ns, tz=UTC]","timestamp[ns, tz=UTC][pyarrow]"
1,product,large_string,large_string[pyarrow]
2,symbol,large_string,large_string[pyarrow]
3,rtype,uint8,uint8[pyarrow]
4,publisher_id,uint16,uint16[pyarrow]
5,instrument_id,uint32,uint32[pyarrow]
6,open,double,double[pyarrow]
7,high,double,double[pyarrow]
8,low,double,double[pyarrow]
9,close,double,double[pyarrow]


,rows
product,
GC,1759671
MGC,1727985


,timestamp_sorted
GC,True
MGC,True


,missing_values
open,0
high,0
low,0
close,0
volume,0


Global physical timestamp order (informational): False
Duplicate (ts_event_utc, product) rows: 0
EXISTING_FORWARD_COLUMNS (8): ['forward_return_5m', 'forward_log_return_5m', 'forward_return_15m', 'forward_log_return_15m', 'forward_return_30m', 'forward_log_return_30m', 'forward_return_60m', 'forward_log_return_60m']


,passed
all_required_columns_present,True
expected_products_only,True
expected_product_row_counts,True
timestamps_sorted_within_each_product,True
duplicate_timestamp_product_rows_zero,True
core_OHLCV_missing_values_zero,True
existing_forward_columns_quarantined,True


## 2.3 Confirm Timestamp and Time-Zone Handling

The physical timezone metadata is validated directly from the saved schema. Each New York timestamp must represent the same instant as its UTC reference, minute alignment must be exact, the saved New York calendar date and clock fields must agree with `ts_event_ny`, and UTC timestamps are not reinterpreted as session clock time.

In [7]:
utc_arrow_type = saved_arrow_schema.field("ts_event_utc").type
ny_arrow_type = saved_arrow_schema.field("ts_event_ny").type

timezone_checks = pd.Series(
    {
        "ts_event_utc_is_timezone_aware_UTC": pa.types.is_timestamp(utc_arrow_type) and utc_arrow_type.tz == "UTC",
        "ts_event_ny_is_timezone_aware_New_York": pa.types.is_timestamp(ny_arrow_type) and ny_arrow_type.tz == RESEARCH_TIMEZONE_NAME,
        "UTC_and_New_York_columns_represent_same_instants": bool(
            research_bars["ts_event_ny"].dt.tz_convert(REFERENCE_TIMEZONE_NAME).eq(research_bars["ts_event_utc"]).all()
        ),
        "UTC_timestamps_are_exactly_minute_aligned": bool(
            research_bars["ts_event_utc"].eq(research_bars["ts_event_utc"].dt.floor("min")).all()
        ),
        "New_York_trade_dates_match_local_calendar_dates": bool(
            research_bars["ts_event_ny"].dt.date.eq(research_bars["trade_date_ny"].dt.date).all()
        ),
        "saved_hour_ny_matches_local_timestamp": bool(
            research_bars["ts_event_ny"].dt.hour.eq(research_bars["hour_ny"]).all()
        ),
        "saved_minute_ny_matches_local_timestamp": bool(
            research_bars["ts_event_ny"].dt.minute.eq(research_bars["minute_ny"]).all()
        ),
    },
    name="passed",
)
display(timezone_checks.to_frame())
display(research_bars[["ts_event_utc", "ts_event_ny", "trade_date_ny", "hour_ny", "minute_ny"]].head(3))

assert timezone_checks.all(), timezone_checks[~timezone_checks].index.tolist()

,passed
ts_event_utc_is_timezone_aware_UTC,True
ts_event_ny_is_timezone_aware_New_York,True
UTC_and_New_York_columns_represent_same_instants,True
UTC_timestamps_are_exactly_minute_aligned,True
New_York_trade_dates_match_local_calendar_dates,True
saved_hour_ny_matches_local_timestamp,True
saved_minute_ny_matches_local_timestamp,True


,ts_event_utc,ts_event_ny,trade_date_ny,hour_ny,minute_ny
0,2021-05-24 00:00:00+00:00,2021-05-23 20:00:00-04:00,2021-05-23 00:00:00,20,0
1,2021-05-24 00:01:00+00:00,2021-05-23 20:01:00-04:00,2021-05-23 00:00:00,20,1
2,2021-05-24 00:02:00+00:00,2021-05-23 20:02:00-04:00,2021-05-23 00:00:00,20,2


## 2.4 Confirm Date Coverage

Coverage is summarized separately for GC and MGC using both UTC event timestamps and New York trading dates. The two product histories are required to share the reported endpoints and trading-date count. Ordinary futures closures are not treated as missing data in this coverage check.

In [8]:
coverage_by_product = (
    research_bars.groupby("product", observed=True)
    .agg(
        rows=("product", "size"),
        first_ts_utc=("ts_event_utc", "min"),
        last_ts_utc=("ts_event_utc", "max"),
        first_trade_date_ny=("trade_date_ny", "min"),
        last_trade_date_ny=("trade_date_ny", "max"),
        trading_dates=("trade_date_ny", "nunique"),
    )
    .sort_index()
)
display(coverage_by_product)

expected_coverage = {
    "first_ts_utc": pd.Timestamp("2021-05-24 00:00:00", tz="UTC"),
    "last_ts_utc": pd.Timestamp("2026-05-22 20:59:00", tz="UTC"),
    "first_trade_date_ny": pd.Timestamp("2021-05-23"),
    "last_trade_date_ny": pd.Timestamp("2026-05-22"),
    "trading_dates": 1_556,
}
coverage_checks = pd.Series(
    {
        name: bool(coverage_by_product[name].eq(expected).all())
        for name, expected in expected_coverage.items()
    },
    name="passed",
)
display(coverage_checks.to_frame())

assert coverage_checks.all(), coverage_checks[~coverage_checks].index.tolist()

,rows,first_ts_utc,last_ts_utc,first_trade_date_ny,last_trade_date_ny,trading_dates
product,,,,,,
GC,1759671,2021-05-24 00:00:00+00:00,2026-05-22 20:59:00+00:00,2021-05-23 00:00:00,2026-05-22 00:00:00,1556
MGC,1727985,2021-05-24 00:00:00+00:00,2026-05-22 20:59:00+00:00,2021-05-23 00:00:00,2026-05-22 00:00:00,1556


,passed
first_ts_utc,True
last_ts_utc,True
first_trade_date_ny,True
last_trade_date_ny,True
trading_dates,True


## 2.5 Confirm Rollover and Exclusion Flags

Rollover, boundary, and research-tradability controls are validated as saved inputs rather than reconstructed. Boolean values are summarized by product, rollover categories and transition labels are enumerated, and day-level roll-distance fields are described. Missing roll distances are retained at leading/trailing coverage boundaries, while `roll_transition_label` is expected to be populated only on roll days. No eligibility filter is applied in this section.

In [9]:
ROLLOVER_AND_EXCLUSION_COLUMNS = [
    "is_roll_day",
    "is_pre_roll_day",
    "is_post_roll_day",
    "roll_window_flag",
    "roll_window_type",
    "roll_group_id",
    "days_since_roll",
    "days_to_next_roll",
    "roll_transition_label",
    "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag",
    "tradable_research_flag",
]
BOOLEAN_CONTROL_COLUMNS = [
    "is_roll_day",
    "is_pre_roll_day",
    "is_post_roll_day",
    "roll_window_flag",
    "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag",
    "tradable_research_flag",
]
missing_control_columns = sorted(set(ROLLOVER_AND_EXCLUSION_COLUMNS).difference(research_bars.columns))
control_missing_values = research_bars[ROLLOVER_AND_EXCLUSION_COLUMNS].isna().sum()

boolean_true_counts = (
    research_bars.groupby("product", observed=True)[BOOLEAN_CONTROL_COLUMNS].sum().astype("int64")
)
boolean_total_counts = research_bars.groupby("product", observed=True).size().astype("int64")
boolean_summary_records = []
for product in EXPECTED_PRODUCTS:
    for column in BOOLEAN_CONTROL_COLUMNS:
        true_count = int(boolean_true_counts.loc[product, column])
        total_count = int(boolean_total_counts.loc[product])
        boolean_summary_records.append(
            {
                "product": product,
                "field": column,
                "false_rows": total_count - true_count,
                "true_rows": true_count,
                "true_rate": true_count / total_count,
            }
        )
boolean_control_summary = pd.DataFrame(boolean_summary_records).set_index(["product", "field"])

roll_window_type_summary = (
    research_bars.groupby(["product", "roll_window_type"], observed=True, dropna=False)
    .size()
    .rename("rows")
    .to_frame()
)
roll_transition_summary = (
    research_bars.groupby(["product", "roll_transition_label"], observed=True, dropna=False)
    .size()
    .rename("rows")
    .to_frame()
)
daily_roll_state = research_bars[
    ["product", "trade_date_ny", "roll_group_id", "days_since_roll", "days_to_next_roll"]
].drop_duplicates()
roll_distance_summary = daily_roll_state.groupby("product", observed=True)[
    ["roll_group_id", "days_since_roll", "days_to_next_roll"]
].agg(["min", "median", "max"])

display(boolean_control_summary)
display(roll_window_type_summary)
display(roll_transition_summary)
display(roll_distance_summary)
display(control_missing_values.rename("missing_values").to_frame())

control_checks = pd.Series(
    {
        "all_rollover_and_exclusion_fields_present": not missing_control_columns,
        "required_boolean_and_categorical_controls_nonmissing": int(
            control_missing_values[BOOLEAN_CONTROL_COLUMNS + ["roll_window_type", "roll_group_id"]].sum()
        ) == 0,
        "roll_transition_label_present_exactly_on_roll_days": bool(
            research_bars["roll_transition_label"].notna().eq(research_bars["is_roll_day"]).all()
        ),
        "roll_distance_boundary_missingness_is_explicit": bool(
            control_missing_values[["days_since_roll", "days_to_next_roll"]].gt(0).all()
        ),
        "roll_window_flag_matches_roll_window_type": bool(
            research_bars["roll_window_flag"].eq(research_bars["roll_window_type"].ne("none")).all()
        ),
        "tradable_research_flag_has_true_and_false_rows": set(
            research_bars["tradable_research_flag"].unique().tolist()
        ) == {False, True},
    },
    name="passed",
)
display(control_checks.to_frame())

assert control_checks.all(), control_checks[~control_checks].index.tolist()

false_rows  true_rows  true_rate
product field                                                          
GC      is_roll_day                       1727634      32037   0.018206
        is_pre_roll_day                   1738275      21396   0.012159
        is_post_roll_day                  1735453      24218   0.013763
        roll_window_flag                  1682020      77651   0.044128
        low_liquidity_active_day_flag     1739487      20184   0.011470
        low_liquidity_warning_flag        1739487      20184   0.011470
        tradable_research_flag              83080    1676591   0.952787
MGC     is_roll_day                       1697290      30695   0.017763
        is_pre_roll_day                   1716268      11717   0.006781
        is_post_roll_day                  1701656      26329   0.015237
        roll_window_flag                  1659244      68741   0.039781
        low_liquidity_active_day_flag     1706300      21685   0.012549
        low_liquidity_warning_flag        1706300      21685   0.012549
        tradable_research_flag              78709    1649276   0.954450

rows
product roll_window_type         
GC      none              1682020
        post_roll_day       24218
        pre_roll_day        21396
        roll_day            32037
MGC     none              1659244
        post_roll_day       26329
        pre_roll_day        11717
        roll_day            30695

rows
product roll_transition_label         
GC      GCG2 -> GCJ2              1376
        GCG3 -> GCJ3              1380
        GCG4 -> GCJ4              1376
        GCG5 -> GCJ5              1319
        GCG6 -> GCJ6              1380
        GCJ2 -> GCM2              1371
        GCJ3 -> GCM3              1379
        GCJ4 -> GCM4              1260
        GCJ5 -> GCM5               120
        GCJ6 -> GCM6              1380
        GCM1 -> GCQ1              1260
        GCM2 -> GCQ2              1228
        GCM3 -> GCQ3              1371
        GCM4 -> GCQ4              1260
        GCM5 -> GCQ5              1380
        GCQ1 -> GCZ1              1260
        GCQ2 -> GCZ2              1260
        GCQ3 -> GCZ3              1375
        GCQ4 -> GCZ4              1379
        GCQ5 -> GCZ5              1379
        GCZ1 -> GCG2              1379
        GCZ2 -> GCG3              1379
        GCZ3 -> GCG4              1380
        GCZ4 -> GCG5              1184
        GCZ5 -> GCG6              1222
        <NA>                   1727634
MGC     MGCG2 -> MGCJ2            1317
        MGCG3 -> MGCJ3            1369
        MGCG4 -> MGCJ4            1361
        MGCG5 -> MGCJ5            1320
        MGCG6 -> MGCJ6            1380
        MGCJ2 -> MGCM2            1371
        MGCJ3 -> MGCM3             120
        MGCJ4 -> MGCM4            1256
        MGCJ5 -> MGCM5            1380
        MGCJ6 -> MGCM6            1380
        MGCM1 -> MGCQ1             100
        MGCM2 -> MGCQ2            1179
        MGCM3 -> MGCQ3            1368
        MGCM4 -> MGCQ4            1258
        MGCM5 -> MGCQ5            1260
        MGCQ1 -> MGCZ1            1325
        MGCQ2 -> MGCZ2            1317
        MGCQ3 -> MGCZ3            1365
        MGCQ4 -> MGCZ4            1380
        MGCQ5 -> MGCZ5            1380
        MGCZ1 -> MGCG2            1364
        MGCZ2 -> MGCG3            1362
        MGCZ3 -> MGCG4            1370
        MGCZ4 -> MGCG5            1185
        MGCZ5 -> MGCG6            1228
        <NA>                   1697290

roll_group_id               days_since_roll                days_to_next_roll               
                  min    median max             min    median  max               min    median  max
product                                                                                            
GC                  1 13.000000  26               0 30.500000  104                 0 31.000000  104
MGC                 1 13.000000  26               0 30.500000  103                 0 31.000000  103

,missing_values
is_roll_day,0
is_pre_roll_day,0
is_post_roll_day,0
roll_window_flag,0
roll_window_type,0
roll_group_id,0
days_since_roll,9982
days_to_next_roll,101823
roll_transition_label,3424924
low_liquidity_active_day_flag,0


,passed
all_rollover_and_exclusion_fields_present,True
required_boolean_and_categorical_controls_nonmissing,True
roll_transition_label_present_exactly_on_roll_days,True
roll_distance_boundary_missingness_is_explicit,True
roll_window_flag_matches_roll_window_type,True
tradable_research_flag_has_true_and_false_rows,True


### Section 2 validation summary

The executable summary below combines the schema, product, timestamp, timezone, coverage, missingness, rollover/tradability, and forward-column quarantine gates. `READY` means the trusted dataset is suitable for the next milestone—construction of the eligible observation frame—but that construction is intentionally outside the scope of this notebook increment.

In [10]:
section_2_validation = pd.concat(
    [schema_checks, timezone_checks, coverage_checks, control_checks],
    keys=["schema_and_content", "timestamps_and_timezones", "coverage", "rollover_and_exclusions"],
).rename("passed")
ready_for_eligible_observation_frame = bool(section_2_validation.all())

display(section_2_validation.to_frame())
print(
    "Section 2 status: "
    + (
        "READY — the trusted GC/MGC table is validated for construction of the eligible observation frame."
        if ready_for_eligible_observation_frame
        else "NOT READY — at least one validation gate failed."
    )
)
print(f"Quarantined outcome-derived columns: {EXISTING_FORWARD_COLUMNS}")

assert ready_for_eligible_observation_frame

passed
schema_and_content       all_required_columns_present                          True
                         expected_products_only                                True
                         expected_product_row_counts                           True
                         timestamps_sorted_within_each_product                 True
                         duplicate_timestamp_product_rows_zero                 True
                         core_OHLCV_missing_values_zero                        True
                         existing_forward_columns_quarantined                  True
timestamps_and_timezones ts_event_utc_is_timezone_aware_UTC                    True
                         ts_event_ny_is_timezone_aware_New_York                True
                         UTC_and_New_York_columns_represent_same_instants      True
                         UTC_timestamps_are_exactly_minute_aligned             True
                         New_York_trade_dates_match_local_calendar_dates       True
                         saved_hour_ny_matches_local_timestamp                 True
                         saved_minute_ny_matches_local_timestamp               True
coverage                 first_ts_utc                                          True
                         last_ts_utc                                           True
                         first_trade_date_ny                                   True
                         last_trade_date_ny                                    True
                         trading_dates                                         True
rollover_and_exclusions  all_rollover_and_exclusion_fields_present             True
                         required_boolean_and_categorical_controls_nonmi...    True
                         roll_transition_label_present_exactly_on_roll_days    True
                         roll_distance_boundary_missingness_is_explicit        True
                         roll_window_flag_matches_roll_window_type             True
                         tradable_research_flag_has_true_and_false_rows        True

Section 2 status: READY — the trusted GC/MGC table is validated for construction of the eligible observation frame.
Quarantined outcome-derived columns: ['forward_return_5m', 'forward_log_return_5m', 'forward_return_15m', 'forward_log_return_15m', 'forward_return_30m', 'forward_log_return_30m', 'forward_return_60m', 'forward_log_return_60m']
